In [1]:
from google.colab import files
uploaded = files.upload()

Saving test.csv.csv to test.csv.csv


In [2]:
from google.colab import files
uploaded = files.upload()

Saving train.csv.csv to train.csv.csv


In [3]:
import pandas as pd
import numpy as npv

In [5]:
train = pd.read_csv('train.csv.csv')
test = pd.read_csv('test.csv.csv')

In [6]:
train.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


In [7]:
train.isnull().sum()

,0
id,0
journal_text,0
ambience_type,0
duration_min,0
sleep_hours,7
energy_level,0
stress_level,0
time_of_day,0
previous_day_mood,15
face_emotion_hint,123


In [8]:
train['emotional_state'].value_counts()
train['intensity'].value_counts()

,count
intensity,
4,277
3,240
5,229
2,228
1,226


In [9]:
import re

def clean_text(text):
    text = str(text).lower()  # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

train['clean_text'] = train['journal_text'].apply(clean_text)
test['clean_text'] = test['journal_text'].apply(clean_text)

In [10]:
train.isnull().sum()

,0
id,0
journal_text,0
ambience_type,0
duration_min,0
sleep_hours,7
energy_level,0
stress_level,0
time_of_day,0
previous_day_mood,15
face_emotion_hint,123


In [11]:
# Numerical columns
num_cols = ['sleep_hours', 'energy_level', 'stress_level', 'duration_min']

for col in num_cols:
    train[col].fillna(train[col].median(), inplace=True)
    test[col].fillna(train[col].median(), inplace=True)

# Categorical columns
cat_cols = ['ambience_type', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality']

for col in cat_cols:
    train[col].fillna('unknown', inplace=True)
    test[col].fillna('unknown', inplace=True)

/tmp/ipykernel_583/2650082456.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].median(), inplace=True)
/tmp/ipykernel_583/2650082456.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [12]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    encoders[col] = le

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

X_text_train = tfidf.fit_transform(train['clean_text'])
X_text_test = tfidf.transform(test['clean_text'])

In [14]:
meta_cols = num_cols + cat_cols

X_meta_train = train[meta_cols].values
X_meta_test = test[meta_cols].values

In [15]:
from scipy.sparse import hstack

X_train = hstack([X_text_train, X_meta_train])
X_test = hstack([X_text_test, X_meta_test])


In [16]:
y_state = train['emotional_state']
y_intensity = train['intensity']

In [17]:
from sklearn.ensemble import RandomForestClassifier

In [18]:
state_model = RandomForestClassifier(n_estimators=100, random_state=42)

state_model.fit(X_train, y_state)

RandomForestClassifier(random_state=42)

In [19]:
state_pred = state_model.predict(X_test)

In [20]:
intensity_model = RandomForestClassifier(n_estimators=100, random_state=42)

intensity_model.fit(X_train, y_intensity)

RandomForestClassifier(random_state=42)

In [21]:
intensity_pred = intensity_model.predict(X_test)

In [22]:
state_proba = state_model.predict_proba(X_test)
state_confidence = state_proba.max(axis=1)

In [23]:
intensity_proba = intensity_model.predict_proba(X_test)
intensity_confidence = intensity_proba.max(axis=1)

In [24]:
confidence = (state_confidence + intensity_confidence) / 2

In [25]:
uncertain_flag = (confidence < 0.6).astype(int)

In [26]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_state, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_tr, y_tr)

print(model.score(X_val, y_val))

0.6458333333333334


In [27]:
def decide_action(state, intensity, stress, energy, time_of_day):

    # HIGH STRESS CASE
    if stress >= 4:
        if energy <= 2:
            return "rest", "now"
        else:
            return "breathing", "now"

    # LOW ENERGY CASE
    if energy <= 2:
        if time_of_day in [0, 1]:  # morning/afternoon (encoded)
            return "light_planning", "within_15_min"
        else:
            return "rest", "tonight"

    # ANXIOUS / NEGATIVE STATES
    if state in ["anxious", "sad", "overwhelmed"]:
        if intensity >= 4:
            return "grounding", "now"
        else:
            return "journaling", "later_today"

    # POSITIVE + HIGH ENERGY
    if state in ["calm", "happy"] and energy >= 3:
        return "deep_work", "now"

    # DEFAULT
    return "pause", "within_15_min"

In [28]:
stress = test['stress_level'].values
energy = test['energy_level'].values
time_of_day = test['time_of_day'].values

In [29]:
state_pred_labels = state_pred  # if already labels

In [30]:
what_to_do = []
when_to_do = []

for i in range(len(test)):
    action, timing = decide_action(
        state_pred_labels[i],
        intensity_pred[i],
        stress[i],
        energy[i],
        time_of_day[i]
    )

    what_to_do.append(action)
    when_to_do.append(timing)

In [32]:
def generate_message(state, action):
    return f"You seem {state}. Let's try {action} to improve your state."

messages = [generate_message(state_pred_labels[i], what_to_do[i]) for i in range(len(test))]

In [33]:
import numpy as np
importances = state_model.feature_importances_

In [34]:
num_text_features = X_text_train.shape[1]
num_meta_features = X_meta_train.shape[1]

In [35]:
text_importance = importances[:num_text_features]
meta_importance = importances[num_text_features:]

In [36]:
print("Text Importance:", np.sum(text_importance))
print("Metadata Importance:", np.sum(meta_importance))

Text Importance: 0.7913383146114059
Metadata Importance: 0.2086616853885941


In [37]:
for col, imp in zip(meta_cols, meta_importance):
    print(col, imp)

sleep_hours 0.02483791789389568
energy_level 0.021282793226130983
stress_level 0.023538722951242274
duration_min 0.029891880320973372
ambience_type 0.022416588577573747
time_of_day 0.019393834631433026
previous_day_mood 0.023361883284257402
face_emotion_hint 0.02573966102008862
reflection_quality 0.018198403482998997


In [38]:
feature_names = tfidf.get_feature_names_out()

top_indices = np.argsort(text_importance)[-10:]

for i in top_indices:
    print(feature_names[i])

calmer
more
not
lighter
tasks
drained
but
nothing
felt
my


In [39]:
X_train_text_only = X_text_train
X_test_text_only = X_text_test

In [40]:
from sklearn.ensemble import RandomForestClassifier

text_only_model = RandomForestClassifier(n_estimators=100, random_state=42)
text_only_model.fit(X_train_text_only, y_state)

RandomForestClassifier(random_state=42)

In [41]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_text_train, y_state, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_tr, y_tr)

text_only_score = model.score(X_val, y_val)
print("Text-only Accuracy:", text_only_score)

Text-only Accuracy: 0.6333333333333333


In [42]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_state, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_tr, y_tr)

combined_score = model.score(X_val, y_val)
print("Text + Metadata Accuracy:", combined_score)

Text + Metadata Accuracy: 0.6375


In [43]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_state, test_size=0.2, random_state=42)

model = RandomForestClassifier()
model.fit(X_tr, y_tr)

y_pred = model.predict(X_val)

In [46]:
val_texts = train.iloc[y_val.index]['journal_text']
print(val_texts)

1178    lowkey felt pretty even, but this was better t...
865     for a while i was more tired than i expected, ...
101     I feel lighter after the forest sounds, like m...
439        Honestly I felt distracted which surprised me.
58      The mountain ambience helped me stop drifting ...
                              ...                        
382     i noticed i could finally concentrate though i...
867                                           felt better
542                           mind was all over the place
1193                                     a little lighter
874     after the session i felt pretty even, but it t...
Name: journal_text, Length: 240, dtype: object


In [47]:
errors = []

for i in range(len(y_val)):
    if y_pred[i] != y_val.iloc[i]:
        errors.append((val_texts.iloc[i], y_val.iloc[i], y_pred[i]))

In [48]:
errors[:10]

[('lowkey felt locked in for a bit, but mountain visuals made it easier to pause.',
  'focused',
  'restless'),
 ('that helped a little', 'calm', 'overwhelmed'),
 ('felt heavy', 'calm', 'overwhelmed'),
 ('got distracted again', 'mixed', 'overwhelmed'),
 ('Honestly still anxious a bit. After some time kinda calm now.',
  'focused',
  'restless'),
 ('okay session ...', 'neutral', 'calm'),
 ('Honestly felt good for a moment. Later it changed mind was all over the place.',
  'calm',
  'restless'),
 ('at first back to normal after. after some time that helped a little.',
  'mixed',
  'overwhelmed'),
 ('back to normal after ...', 'overwhelmed', 'restless'),
 ('not gonna lie i felt pretty overloaded. cafe ambience weirdly helped. not sure why but it shifted.',
  'overwhelmed',
  'mixed')]

In [49]:
output = pd.DataFrame({
    'id': test['id'],
    'predicted_state': state_pred,
    'predicted_intensity': intensity_pred,
    'confidence': confidence,
    'uncertain_flag': uncertain_flag,
    'what_to_do': what_to_do,
    'when_to_do': when_to_do
})

output.to_csv('predictions.csv', index=False)

In [50]:
from google.colab import files
files.download('predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>